# Additional Metrics

In [1]:
from evals import Dataset

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


In [3]:
ds = Dataset("test")
data = ds.load_results()
res = data.get("datasets/test/04_results/google_gemini-2.5-flash_baseline_2161608a-ae80-4fb7-9f2e-33de91e0316d.json")

In [4]:
for session in res.sessions:
    for conv in session.conversation:
        expected_output = conv.answer
        actual_output = conv.model_response

## Rouge-L

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
result = scorer.score(target=expected_output, prediction=actual_output)
print(result["rougeL"])  # precision, recall, fmeasure

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mdeberta-v3-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/mdeberta-v3-base/a0484667b22365f84929a935b5e50a51f71f159d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mdeberta-v3-base/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/mdeberta-v3-base/a0484667b22365f84929a935b5e50a51f71f159d/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request:

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/mdeberta-v3-base/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/commits/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/discussions?p=0 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/commits/refs%2Fpr%2F5 "HTTP/1.1 200 OK"
DebertaV2Model LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask

OverflowError: int too big to convert

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/microsoft/mdeberta-v3-base/xet-read-token/faf660b811036d7b6dfc53aacfe5281447d4e2c6 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

## Self-consistency

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(outputs)  # outputs = liste med N svar
sim_matrix = cosine_similarity(embeddings)
# Gjennomsnitt av upper triangle (ekskl. diagonal)
n = len(outputs)
mask = np.triu(np.ones((n, n), dtype=bool), k=1)
consistency_score = sim_matrix[mask].mean()